In [41]:
%load_ext autoreload
%autoreload 2

In [31]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [ ]:
pitcher_games = pd.read_csv("../data/processed/pitcher_games_features_base.csv")

model = joblib.load("../models/xgboost.pkl")

with open("../models/model_features.json") as f:
    feature_cols = json.load(f)

In [5]:
pitcher_games["game_date"] = pd.to_datetime(pitcher_games["game_date"])

pitcher_games = pitcher_games.sort_values(["pitcher", "game_date"])

In [9]:
cutoff = pd.Timestamp("2024-08-01")

backtest_train = pitcher_games[pitcher_games["game_date"] < cutoff].copy()

backtest_test = pitcher_games[pitcher_games["game_date"] >= cutoff].copy()

In [12]:
backtest_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        (
            "xgb",XGBRegressor(
                objective="reg:squarederror",
                colsample_bytree=0.8,
                learning_rate=0.05,
                max_depth=3,
                n_estimators=100,
                subsample=1.0,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [13]:
X_train = backtest_train[feature_cols]
y_train = backtest_train["strikeouts"]

X_test = backtest_test[feature_cols]
y_test = backtest_test["strikeouts"]

In [14]:
backtest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](49,)","['rest_days','max_times_through_order','k_last3',..., 'rolling3_called_strikes','rolling5_called_strikes', 'season_called_strikes']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,49
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata an

In [15]:
backtest_test["predicted_strikeouts"] = (backtest_model.predict(X_test))

In [16]:
backtest_test[["game_date", "player_name", "strikeouts", "predicted_strikeouts"]].head(20)

,game_date,player_name,strikeouts,predicted_strikeouts
10,2024-08-21,"Verlander, Justin",6,4.946564
11,2024-08-27,"Verlander, Justin",3,5.134508
12,2024-09-02,"Verlander, Justin",3,4.908831
13,2024-09-08,"Verlander, Justin",0,4.973015
14,2024-09-14,"Verlander, Justin",2,4.835873
15,2024-09-20,"Verlander, Justin",4,4.568579
16,2024-09-28,"Verlander, Justin",5,4.994508
39,2024-08-01,"Morton, Charlie",6,4.912133
40,2024-08-08,"Morton, Charlie",3,3.160828
41,2024-08-13,"Morton, Charlie",8,4.908707


In [17]:
backtest_mae = mean_absolute_error(y_test, backtest_test["predicted_strikeouts"])

backtest_rmse = np.sqrt(mean_squared_error(y_test, backtest_test["predicted_strikeouts"]))

backtest_r2 = r2_score(y_test, backtest_test["predicted_strikeouts"])

print(f"Backtest MAE: {backtest_mae:.3f}")
print(f"Backtest RMSE: {backtest_rmse:.3f}")
print(f"Backtest R²: {backtest_r2:.3f}")

Backtest MAE: 1.729
Backtest RMSE: 2.157
Backtest R²: 0.250


In [18]:
backtest_test["prediction_rank"] = (backtest_test.groupby("game_date")["predicted_strikeouts"].rank(ascending=False, method="first"))

In [19]:
backtest_test[
    backtest_test["game_date"] == backtest_test["game_date"].iloc[0]
][
    [
        "game_date",
        "player_name",
        "predicted_strikeouts",
        "strikeouts",
        "prediction_rank"
    ]
].sort_values("prediction_rank")

,game_date,player_name,predicted_strikeouts,strikeouts,prediction_rank
2311,2024-08-21,"Flaherty, Jack",6.082962,5,1.0
3456,2024-08-21,"Gilbert, Logan",5.868623,7,2.0
1849,2024-08-21,"Manaea, Sean",5.754618,9,3.0
4777,2024-08-21,"Pepiot, Ryan",5.665019,5,4.0
2507,2024-08-21,"Webb, Logan",5.582415,6,5.0
4672,2024-08-21,"Rodríguez, Yariel",5.265855,6,6.0
1414,2024-08-21,"Fried, Max",5.143272,4,7.0
1132,2024-08-21,"Nola, Aaron",5.095207,5,8.0
845,2024-08-21,"Taillon, Jameson",5.093792,5,9.0
182,2024-08-21,"Gibson, Kyle",5.048908,5,10.0


In [20]:
backtest_test["decision"] = np.where(backtest_test["predicted_strikeouts"] >= 5.0, "START", "SIT")

In [ ]:
backtest_test[["game_date", "player_name", "predicted_strikeouts", "strikeouts", "decision"]].head(20)

,game_date,player_name,predicted_strikeouts,strikeouts,decision
10,2024-08-21,"Verlander, Justin",4.946564,6,SIT
11,2024-08-27,"Verlander, Justin",5.134508,3,START
12,2024-09-02,"Verlander, Justin",4.908831,3,SIT
13,2024-09-08,"Verlander, Justin",4.973015,0,SIT
14,2024-09-14,"Verlander, Justin",4.835873,2,SIT
15,2024-09-20,"Verlander, Justin",4.568579,4,SIT
16,2024-09-28,"Verlander, Justin",4.994508,5,SIT
39,2024-08-01,"Morton, Charlie",4.912133,6,SIT
40,2024-08-08,"Morton, Charlie",3.160828,3,SIT
41,2024-08-13,"Morton, Charlie",4.908707,8,SIT


In [ ]:
decision_summary = (backtest_test.groupby("decision")["strikeouts"].agg(games="count", average_actual_k="mean", median_actual_k="median"))

decision_summary

,games,average_actual_k,median_actual_k
decision,,,
SIT,892,3.952915,4.0
START,750,5.834667,6.0


In [23]:
backtest_test["actual_success"] = (backtest_test["strikeouts"] >= 5)

decision_accuracy = (backtest_test.groupby("decision")["actual_success"].mean())

decision_accuracy

decision
SIT      0.387892
START    0.694667
Name: actual_success, dtype: float64

In [24]:
backtest_test[["game_date", "player_name", "predicted_strikeouts", "strikeouts", "decision"]].sort_values("predicted_strikeouts", ascending=False).head(10)

,game_date,player_name,predicted_strikeouts,strikeouts,decision
1165,2024-08-12,"Snell, Blake",8.622433,11,START
1168,2024-08-30,"Snell, Blake",7.694146,8,START
1172,2024-09-22,"Snell, Blake",7.346352,9,START
1163,2024-08-02,"Snell, Blake",7.329849,11,START
2953,2024-08-18,"Valdez, Framber",7.318032,9,START
1164,2024-08-07,"Snell, Blake",7.314189,8,START
1166,2024-08-18,"Snell, Blake",7.296875,10,START
3511,2024-08-07,"Skubal, Tarik",7.277342,9,START
2146,2024-09-11,"King, Michael",7.259226,6,START
2952,2024-08-12,"Valdez, Framber",7.221440,9,START


In [25]:
backtest_test["prediction_error"] = (backtest_test["strikeouts"] - backtest_test["predicted_strikeouts"])

backtest_test[["game_date", "player_name", "predicted_strikeouts", "strikeouts", "prediction_error"]].sort_values("prediction_error", key=abs, ascending=False).head(10)

,game_date,player_name,predicted_strikeouts,strikeouts,prediction_error
3683,2024-08-24,"Francis, Bowden",4.689429,12,7.310571
3816,2024-09-27,"Detmers, Reid",4.835805,12,7.164195
4782,2024-09-18,"Pepiot, Ryan",5.111944,12,6.888056
4237,2024-09-13,"Marsh, Alec",4.394040,11,6.605960
4436,2024-08-10,"Arrighetti, Spencer",6.420976,13,6.579024
5020,2024-09-19,"Pfaadt, Brandon",5.445427,12,6.554573
2054,2024-09-18,"Ober, Bailey",5.517276,12,6.482724
761,2024-08-31,"Kikuchi, Yusei",5.757446,12,6.242554
5206,2024-09-29,"Birdsong, Hayden",4.888429,11,6.111571
4435,2024-08-04,"Arrighetti, Spencer",5.914933,12,6.085067


In [26]:
overall_avg_k = backtest_test["strikeouts"].mean()

start_avg_k = (backtest_test.loc[backtest_test["decision"] == "START", "strikeouts"].mean())

sit_avg_k = (backtest_test.loc[backtest_test["decision"] == "SIT", "strikeouts"].mean())

print(f"Overall average K: {overall_avg_k:.3f}")
print(f"START average K: {start_avg_k:.3f}")
print(f"SIT average K: {sit_avg_k:.3f}")
print(f"START advantage: " f"{start_avg_k - overall_avg_k:.3f} Ks")

Overall average K: 4.812
START average K: 5.835
SIT average K: 3.953
START advantage: 1.022 Ks


In [28]:
backtest_test.to_csv("../data/processed/model_evaluation_stats.csv", index=False)

stage5_summary = pd.DataFrame({
    "Metric": [
        "Backtest MAE",
        "Backtest RMSE",
        "Backtest R2",
        "START Games",
        "SIT Games",
        "START Avg K",
        "SIT Avg K",
        "START 5+ K Rate",
        "SIT 5+ K Rate"
    ],
    "Value": [
        backtest_mae,
        backtest_rmse,
        backtest_r2,
        750,
        892,
        5.834667,
        3.952915,
        0.694667,
        0.387892
    ]
})

stage5_summary.to_csv("../data/processed/stage5_results.csv", index=False)